# Fluorescence Lifetime Imaging (FLIM) Analysis Pipeline

## Overview
This script calculates mean fluorescence lifetime (τ) images from time-tagged time-resolved (TTTR) data. Fluorescence lifetime provides complementary information to intensity measurements and is particularly valuable for FRET analysis, as FRET reduces donor lifetime proportional to FRET efficiency.

## Input/Output
- **Input**: 
  - `*.ptu` files (TTTR data from samples)
  - `IRF.ptu` (Instrument Response Function for deconvolution)
- **Output**: 
  - `*_mean tau green.tif` (mean lifetime images)

## Experimental Context
- **FLIM-FRET**: Fluorescence lifetime changes upon FRET occurrence
- **Donor channel analysis**: Green channel (donor) lifetime measurement
- **IRF correction**: Accounts for instrument temporal resolution

## Configuration Parameters
- **Channels**: Green s+p polarization (0,3) for donor detection
- **Time window**: 0-25 ns (prompt excitation window)
- **Photon threshold**: Minimum 15 photons per pixel for reliable lifetime calculation
- **Processing**: Frame stacking enabled for improved statistics

## Mathematical Background
- **Mean lifetime**: τ = ∫ t·I(t)dt / ∫ I(t)dt (intensity-weighted average)
- **IRF deconvolution**: Corrects for laser pulse delay
- **FRET sensitivity**: τ_DA < τ_D when FRET occurs

## Processing Steps
1. Load sample TTTR data and IRF reference
2. Filter data by channel and time window
3. Calculate pixel-wise mean lifetime with IRF correction
4. Apply photon count threshold for statistical reliability
5. Generate and save lifetime images

In [1]:
# Import required libraries
import tttrlib                  # TTTR data processing and lifetime analysis
import pylab as plt             # Matplotlib plotting interface
import skimage as ski           # Image processing toolkit
from skimage import io          # Image I/O operations
import numpy as np              # Numerical operations
from mpl_toolkits.axes_grid1 import make_axes_locatable   # Subplot formatting
import glob                     # File pattern matching
import os                       # Operating system interface

## CONFIGURATION AND FILE PATHS

In [2]:
# Define input file pattern for batch processing
path ='C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_Files/Cells/*/*.ptu'

In [3]:
# Path to Instrument Response Function (IRF) calibration file
# IRF characterizes the temporal resolution of the detection system
filename_irf =  'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_Files/Calibration/IRF.ptu'
irf = tttrlib.TTTR(filename_irf)

In [4]:
# Define detection parameters
channel = [0, 3]       # Green channels: s-polarized (0) and p-polarized (3)
time_range = 0, 2500   # Prompt time window in channel numbers (donor excitation)

## IRF PREPARATION AND CALIBRATION

In [5]:
# Generate IRF mask to select relevant channels and time ranges
# The IRF corrects for instrument temporal broadening effects

mask_irf = tttrlib.TTTRMask()
mask_irf.select_channels(irf, channel)                 # Same channels as sample
mask_irf.select_microtime_ranges(irf, [time_range])    # Same time window as sample
tttr_irf = irf[mask_irf.indices]                       # Extract filtered IRF data

## BATCH PROCESSING OF LIFETIME DATA

In [6]:
# Process each sample file matching the specified pattern
for file in glob.glob(path):
    # Extract base filename for output naming
    filename = os.path.abspath(file).split(".")[0]
    
    # Load sample TTTR data
    data = tttrlib.TTTR(file)
    
    # Extract green channel data (donor fluorescence)
    tttr_green = data.get_tttr_by_channel(channel)
    
    # Initialize CLSM image container with specified parameters
    clsm_green = tttrlib.CLSMImage(
        tttr_data=data, 
        channels=channel, 
        micro_time_ranges=[time_range]
    )
    
    print('Processing....' + filename)

    # MEAN LIFETIME CALCULATION
    mean_tau = clsm_green.get_mean_lifetime(
        data,
        minimum_number_of_photons=15,     # Statistical threshold: require ≥15 photons/pixel   
        stack_frames=True,                # Combine all time frames for better statistics
        tttr_irf=tttr_irf                 # Apply IRF deconvolution correction
    )
    
    # The result is a lifetime image where each pixel value represents
    # the mean fluorescence lifetime in that spatial location
    
    # VISUALIZATION AND DATA EXPORT
    
    # Create lifetime image visualization
    ax = plt.subplot()
    # Use 'viridis' colormap: purple (short lifetime) to yellow (long lifetime)
    im = ax.imshow(mean_tau[0], cmap='viridis')
    ax.set_title(f'Mean Fluorescence Lifetime\n{os.path.basename(filename)}')
    ax.set_xlabel('X Position (pixels)')
    ax.set_ylabel('Y Position (pixels)')

    # Add colorbar with proper formatting
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = plt.colorbar(im, cax=cax)
    cbar.set_label('Lifetime (ns)', rotation=270, labelpad=15)
    
    # Save visualization (optional - currently commented out)
    # plt.savefig(filename + '_lifetime_plot.png', dpi=150, bbox_inches='tight')
    # plt.show()

    # Export lifetime image as TIFF file (without colorbar)
    # This preserves the quantitative lifetime values for further analysis
    ski.io.imsave(filename + '_mean tau green.tif', mean_tau[0])
    
    # Close plot to free memory
    plt.close()

Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\cytosolic\eGFP_44
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_66
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_68
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_70
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_72
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_74
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_76
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\Donly\DO_24
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\Donly\high_Donly_32
Processing....C:\Users\Kather

## OUTPUT INTERPRETATION GUIDE

Generated lifetime images contain the following information:

PIXEL VALUES:
- Each pixel represents mean fluorescence lifetime in nanoseconds
- Typical donor lifetimes: 2-4 ns (depending on fluorophore)
- FRET reduces lifetime: shorter τ indicates higher FRET efficiency
- Pixels with <15 photons are excluded (set to 0 or background)

COLOR MAPPING (viridis colormap):
- Purple/Blue: Short lifetimes (high FRET, strong interactions)
- Green: Intermediate lifetimes (moderate FRET)
- Yellow: Long lifetimes (low FRET, minimal interactions)

BIOLOGICAL INTERPRETATION:
- Uniform short lifetime: Strong, homogeneous protein interactions
- Heterogeneous lifetime: Spatially varying interaction strengths  
- Long lifetime regions: Areas with minimal protein interactions
- Lifetime gradients: Interaction strength gradients within cells

ANALYSIS APPLICATIONS:
- FRET efficiency mapping: E = 1 - (τ_DA/τ_D)
- Protein interaction quantification
- Subcellular localization of interactions
- Comparative analysis between conditions
- Complementary to intensity-based FRET measurements

QUALITY CONTROL:
- Check photon count thresholds in low-signal regions
- Verify IRF correction is appropriate for your system
- Compare with intensity images for consistency
- Monitor for photobleaching effects in time series